In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

/Users/martindufour/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Function 2
print('Function 2')
func2_inputs = np.load('./initial_data/function_2/initial_inputs.npy')
print(func2_inputs)

func2_outputs = np.load('./initial_data/function_2/initial_outputs.npy')
print(func2_outputs)
print('/n')

Function 2
[[0.66579958 0.12396913]
 [0.87779099 0.7786275 ]
 [0.14269907 0.34900513]
 [0.84527543 0.71112027]
 [0.45464714 0.29045518]
 [0.57771284 0.77197318]
 [0.43816606 0.68501826]
 [0.34174959 0.02869772]
 [0.33864816 0.21386725]
 [0.70263656 0.9265642 ]]
[ 0.53899612  0.42058624 -0.06562362  0.29399291  0.21496451  0.02310555
  0.24461934  0.03874902 -0.01385762  0.61120522]
/n


In [3]:
# Week 1 Input and Output Data
week_1_inputs = [ np.array([0.155793, 0.528435]), np.array([0.224164, 0.812385]), np.array([0.830919, 0.158523, 0.528406]), np.array([0.912533, 0.052672, 0.771239, 0.219812]), np.array([0.234189, 0.83648 , 0.884484, 0.873516]), np.array([0.490808, 0.618683, 0.277824, 0.900494, 0.106596]), np.array([0.067896, 0.486672, 0.255422, 0.215118, 0.427428, 0.72097 ]), np.array([0.061447, 0.062956, 0.029929, 0.036786, 0.407935, 0.795055, 0.496307,0.888085]) ]
week_1_outputs = [np.float64(1.5311489892413605e-58), np.float64(0.04614596805685454), np.float64(-0.04591165123945737), np.float64(-24.387232512869755), np.float64(1049.4420694211206), np.float64(-0.849252655626155), np.float64(1.3793294734939503), np.float64(9.598780741169)]

week_2_inputs = [np.array([0.946399, 0.18731 ]), np.array([0.712637, 0.921564]), np.array([0.765104, 0.052672, 0.438597]), np.array([0.380965, 0.771701, 0.089479, 0.536968]), np.array([0.219189, 0.85148 , 0.874484, 0.883516]), np.array([0.114755, 0.697421, 0.354179, 0.887624, 0.589139]), np.array([0.070896, 0.484672, 0.259422, 0.216118, 0.427428, 0.72297 ]), np.array([0.062447, 0.061956, 0.030929, 0.035786, 0.408935, 0.794055, 0.497307, 0.887085])]
week_2_outputs = [np.float64(2.338449488843798e-206), np.float64(0.5728372778652475), np.float64(-0.10235480701941752), np.float64(-13.368584815829887), np.float64(1109.9883069580462), np.float64(-1.492909868264592), np.float64(1.3944037721068683), np.float64(9.598978827169)]

week_3_inputs = [np.array([0.583738, 0.706798]), np.array([0.699637, 0.928564]), np.array([0.123457, 0.876543, 0.5     ]), np.array([0.230785, 0.914568, 0.102938, 0.657322]), np.array([0.214189, 0.85648 , 0.869484, 0.888516]), np.array([0.051235, 0.987654, 0.43211 , 0.123457, 0.765432]), np.array([0.071896, 0.483672, 0.261422, 0.217118, 0.426428, 0.72497 ]), np.array([0.063447, 0.060956, 0.031929, 0.034786, 0.409935, 0.793055, 0.498307, 0.886085])]
week_3_outputs = [np.float64(9.817718646044271e-07), np.float64(0.49978778689465564), np.float64(-0.04054152149606349), np.float64(-21.112987453792396), np.float64(1132.5255136709882), np.float64(-2.4679805862566795), np.float64(1.4059619293543582), np.float64(9.599155713169)]

week_4_inputs = [np.array([0.867072, 0.913241]), np.array([0.83604 , 0.696071]), np.array([0.447658, 0.395195, 0.505344]), np.array([0.422706, 0.385497, 0.37529 , 0.410833]), np.array([0.157706, 0.912326, 0.830158, 0.925715]), np.array([0.275401, 0.      , 0.568079, 1.      , 0.121326]), np.array([0.124679, 0.389027, 0.389012, 0.23686 , 0.379036, 0.802247]), np.array([0.077447, 0.21029 , 0.114032, 0.159535, 0.695924, 0.531316,0.178973, 0.57168 ])]
week_4_outputs = [np.float64(-1.0508862613282513e-96), np.float64(0.20657541488475104), np.float64(-0.03229682155733878), np.float64(0.4964978124935766), np.float64(1445.380906735266), np.float64(-0.8152779914672599), np.float64(2.0217812896063525), np.float64(9.987130922543)]

week_5_inputs = [np.array([0.065052, 0.948886]), np.array([0.388677, 0.271349]), np.array([1.      , 0.136717, 0.850593]), np.array([0.908266, 0.239562, 0.144895, 0.489453]), np.array([0.115614, 0.955641, 0.820859, 0.938174]), np.array([0.568442, 0.      , 1.      , 1.      , 1.      ]), np.array([0.134015, 0.028783, 0.755137, 0.62031 , 0.70408 , 0.212964]), np.array([0.127008, 0.292876, 0.06967 , 0.277582, 0.553407, 0.547258,0.220835, 0.443146])]
week_5_outputs = [np.float64(2.0262778967114778e-283), np.float64(0.016418658339648333), np.float64(-0.054388754089278846), np.float64(-17.161465002411145), np.float64(1779.8600577462366), np.float64(-1.9906490554141107), np.float64(0.052467603616080494), np.float64(9.9149654065019)]

week_6_inputs = [np.array([0.17701 , 0.088703]), np.array([0.593592, 0.679102]), np.array([0.7536, 1.    , 1.    ]), np.array([0.370348, 0.379959, 0.430216, 0.444351]), np.array([0.169493, 0.556801, 0.936155, 0.69603 ]), np.array([0.366464, 0.316099, 1.      , 1.      , 0.      ]), np.array([0.123574, 0.270452, 0.482301, 0.208163, 0.330398, 0.872733]), np.array([0.3191  , 0.828915, 0.037008, 0.59627 , 0.230009, 0.120567,0.076953, 0.696289])]
week_6_outputs = [np.float64(-2.3725238219366144e-119), np.float64(0.06836721478932847), np.float64(-0.48310415434111403), np.float64(0.18076540708623456), np.float64(282.83820524691396), np.float64(-0.8214281898088153), np.float64(2.075605759888563), np.float64(8.8803427965834)]

week_7_inputs = [np.array([0.065052, 0.948886]), np.array([0.140924, 0.802197]), np.array([1., 1., 0.]), np.array([0.32078 , 0.186519, 0.040775, 0.590893]), np.array([0.548734, 0.691895, 0.651961, 0.224269]), np.array([0.368433, 0.      , 1.      , 1.      , 0.428022]), np.array([0.139689, 0.317433, 0.463608, 0.250431, 0.325485, 0.810756]), np.array([0.      , 0.202823, 0.231703, 0.      , 1.      , 1.      ,
       0.288474, 1.      ])]
week_7_outputs = [np.float64(2.0262778967114778e-283), np.float64(-0.10826299524356352), np.float64(-0.16354530625442043), np.float64(-11.58523458824008), np.float64(1.9931553503870212), np.float64(-1.2796687884296385), np.float64(2.462201676843452), np.float64(9.622023932692)]

week_8_inputs = [np.array([0.000788, 0.033717]), np.array([0.914607, 0.789979]), np.array([0.317253, 0.002183, 0.963506]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.078217, 0.973099, 0.868006, 0.933352]), np.array([0.426158, 0.348959, 0.616644, 0.692851, 0.024814]), np.array([0.269889, 0.346609, 0.467584, 0.256632, 0.302654, 0.800092]), np.array([0.066269, 0.029193, 0.143059, 0.209133, 0.840619, 0.604919,
       0.230297, 0.701468])]
week_8_outputs = [np.float64(1.5608341712501477e-228), np.float64(0.0347797753016137), np.float64(-0.3756702789549372), np.float64(-33.661790988299735), np.float64(2151.3700669834334), np.float64(-0.23000336822278494), np.float64(2.4516632535923746), np.float64(9.9644234438351)]

week_9_inputs = [np.array([0.000924, 0.003116]), np.array([0.914607, 0.789979]), np.array([0.047574, 0.998325, 0.999125]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.080698, 0.971918, 0.842327, 0.954471]), np.array([0.942348, 0.037756, 0.105925, 0.010348, 0.997595]), np.array([0.090198, 0.680559, 0.851748, 0.080076, 0.298474, 0.701193]), np.array([0.95983 , 0.002073, 0.011779, 0.227207, 0.571611, 0.031204,
       0.24468 , 0.055697])]
week_9_outputs = [np.float64(7.25285761175276e-246), np.float64(0.05670257386671321), np.float64(-0.48158498276260003), np.float64(-33.661790988299735), np.float64(2156.522419821577), np.float64(-3.026993482898997), np.float64(0.6117443854593981), np.float64(8.1721431718416)]

week_10_inputs = [np.array([0.999878, 0.001995]), np.array([0.41611 , 0.666998]), np.array([0.001378, 0.999625, 0.642937]), np.array([0.027746, 0.654937, 0.999652, 0.024674]), np.array([0.109191, 0.976767, 0.866838, 0.873854]), np.array([0.944032, 0.013284, 0.976966, 0.023315, 0.997903]), np.array([0.167912, 0.257921, 0.480203, 0.472529, 0.254657, 0.778833]), np.array([0.116951, 0.002708, 0.212486, 0.121597, 0.986678, 0.488014,
       0.152731, 0.379341])]
week_10_outputs = [np.float64(0.0), np.float64(0.1625184332362701), np.float64(-0.13457392031599885), np.float64(-30.133085064011215), np.float64(1769.6901405375768), np.float64(-2.7724961974553874), np.float64(1.9208154093840464), np.float64(9.9296060847489)]

In [4]:
# Function 2
print('Function 2')
# Load inputs from previous run
# Loads initial data
week_0_func2_inputs = np.load('./initial_data/function_2/initial_inputs.npy')
week_0_func2_outputs = np.load('./initial_data/function_2/initial_outputs.npy')

print(f'Shape of initial input data: {week_0_func2_inputs.shape}')
print(f'Shape of initial output data: {week_0_func2_outputs.shape}')

print(f'Week 1 inputs: {week_1_inputs[1]}')
print(f'Week 2 inputs: {week_2_inputs[1]}')
print(f'Week 3 inputs: {week_3_inputs[1]}')
print(f'Week 4 inputs: {week_4_inputs[1]}')
print(f'Week 5 inputs: {week_5_inputs[1]}')
print(f'Week 6 inputs: {week_6_inputs[1]}')
print(f'Week 7 inputs: {week_7_inputs[1]}')
print(f'Week 8 inputs: {week_8_inputs[1]}')
print(f'Week 9 inputs: {week_9_inputs[1]}')
print(f'Week 10 inputs: {week_10_inputs[1]}')

combined_func2_inputs = np.vstack([
    week_0_func2_inputs,
    week_1_inputs[1],
    week_2_inputs[1],
    week_3_inputs[1],
    week_4_inputs[1],
    week_5_inputs[1],
    week_6_inputs[1],
    week_7_inputs[1],
    week_8_inputs[1],
    week_9_inputs[1],
    week_10_inputs[1]
])
print(f'Number of input data points: {len(combined_func2_inputs)}')
print('Combined input data')
print(combined_func2_inputs)

# Load outputs from previous run
week_func2_output = week_1_outputs[1]
combined_func2_outputs = np.concatenate([
    week_0_func2_outputs,
    [week_1_outputs[1]],
    [week_2_outputs[1]],
    [week_3_outputs[1]],
    [week_4_outputs[1]],
    [week_5_outputs[1]],
    [week_6_outputs[1]],
    [week_7_outputs[1]],
    [week_8_outputs[1]],
    [week_9_outputs[1]],
    [week_10_outputs[1]]
])
print(f'Number of output data points: {len(combined_func2_outputs)}')
print('Combined output data')
print(combined_func2_outputs)

Function 2
Shape of initial input data: (10, 2)
Shape of initial output data: (10,)
Week 1 inputs: [0.224164 0.812385]
Week 2 inputs: [0.712637 0.921564]
Week 3 inputs: [0.699637 0.928564]
Week 4 inputs: [0.83604  0.696071]
Week 5 inputs: [0.388677 0.271349]
Week 6 inputs: [0.593592 0.679102]
Week 7 inputs: [0.140924 0.802197]
Week 8 inputs: [0.914607 0.789979]
Week 9 inputs: [0.914607 0.789979]
Week 10 inputs: [0.41611  0.666998]
Number of input data points: 20
Combined input data
[[0.66579958 0.12396913]
 [0.87779099 0.7786275 ]
 [0.14269907 0.34900513]
 [0.84527543 0.71112027]
 [0.45464714 0.29045518]
 [0.57771284 0.77197318]
 [0.43816606 0.68501826]
 [0.34174959 0.02869772]
 [0.33864816 0.21386725]
 [0.70263656 0.9265642 ]
 [0.224164   0.812385  ]
 [0.712637   0.921564  ]
 [0.699637   0.928564  ]
 [0.83604    0.696071  ]
 [0.388677   0.271349  ]
 [0.593592   0.679102  ]
 [0.140924   0.802197  ]
 [0.914607   0.789979  ]
 [0.914607   0.789979  ]
 [0.41611    0.666998  ]]
Number of ou

In [5]:
### ====== OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 2 ======
# Import the OptunaBayesianOptimizer class
import sys
sys.path.insert(0, './bayesian_optimization_challenge-md')
from bo_optuna import OptunaBayesianOptimizer

# Create optimizer instance with initial Function 2 data
print("=" * 60)
print("OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 2")
print("=" * 60)

X_func2_initial = week_0_func2_inputs
y_func2_initial = week_0_func2_outputs
bounds_func2 = [(0, 1)] * X_func2_initial.shape[1]

optimizer_func2 = OptunaBayesianOptimizer(
    X_initial=X_func2_initial,
    y_initial=y_func2_initial,
    bounds=bounds_func2,
    optimize_hp=True,  # Enable hyperparameter tuning
    random_state=42,
    acquisition="ucb"
)

print(f"\nInitial training data shape: X={optimizer_func2.X_train.shape}, y={optimizer_func2.y_train.shape}")
print(f"Initial best observation: {optimizer_func2.get_best_observation()[1]:.6e}")


OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 2

Initial training data shape: X=(10, 2), y=(10,)
Initial best observation: 6.112052e-01


In [6]:
# Run Optuna-based BO for 8 weeks (8 iterations)
print("\nRunning Optuna-based BO for 8 iterations...")
print("-" * 60)

# Get all weekly data
weekly_data = [
    (week_1_inputs[1], week_1_outputs[1]),
    (week_2_inputs[1], week_2_outputs[1]),
    (week_3_inputs[1], week_3_outputs[1]),
    (week_4_inputs[1], week_4_outputs[1]),
    (week_5_inputs[1], week_5_outputs[1]),
    (week_6_inputs[1], week_6_outputs[1]),
    (week_7_inputs[1], week_7_outputs[1]),
    (week_8_inputs[1], week_8_outputs[1]),
    (week_9_inputs[1], week_9_outputs[1]),
    (week_10_inputs[1], week_10_outputs[1])
]

optuna_proposals_func2 = []
manual_best_func2 = week_0_func2_outputs.max()
optuna_best_func2 = y_func2_initial.max()

for week, (x_actual, y_actual) in enumerate(weekly_data, start=1):
    print(f"\nWeek {week}:")
    print(f"  Actual observation: y = {y_actual:.6e}")
    
    # Get Optuna proposal
    proposals = optimizer_func2.optimize(
        n_iterations=1,
        optimize_hp_every=1 if week % 2 == 0 else 0,  # Tune HP every other week
        optimize_hp_n_trials=30,
        acq_n_trials=100,
        verbose=True
    )
    
    x_proposed = proposals[0]
    optuna_proposals_func2.append(x_proposed)
    
    # Update optimizer with actual observation
    optimizer_func2.update(x_actual, y_actual)
    
    # Track best values
    manual_best_func2 = max(manual_best_func2, y_actual)
    optuna_best_func2 = max(optuna_best_func2, y_actual)
    
    print(f"  Best so far (Optuna): {optuna_best_func2:.6e}")

print("\n" + "=" * 60)
print("OPTUNA-BASED BO COMPLETED FOR FUNCTION 2")
print("=" * 60)



Running Optuna-based BO for 8 iterations...
------------------------------------------------------------

Week 1:
  Actual observation: y = 4.614597e-02


[I 2026-04-23 19:22:47,276] A new study created in memory with name: no-name-e6e6d6b9-3a59-4c33-ab23-5c262cfe18f4
[I 2026-04-23 19:22:47,279] Trial 0 finished with value: 4.817142548465194 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162}. Best is trial 0 with value: 4.817142548465194.
[I 2026-04-23 19:22:47,281] Trial 1 finished with value: 3.0932504656147057 and parameters: {'x0': 0.7319939418114051, 'x1': 0.5986584841970366}. Best is trial 0 with value: 4.817142548465194.
[I 2026-04-23 19:22:47,284] Trial 2 finished with value: 3.567411930100772 and parameters: {'x0': 0.15601864044243652, 'x1': 0.15599452033620265}. Best is trial 0 with value: 4.817142548465194.
[I 2026-04-23 19:22:47,286] Trial 3 finished with value: 4.982903268182547 and parameters: {'x0': 0.05808361216819946, 'x1': 0.8661761457749352}. Best is trial 3 with value: 4.982903268182547.
[I 2026-04-23 19:22:47,287] Trial 4 finished with value: 1.7087259999407458 and parameters: {'x0': 0.6011150117432

[Iteration 0] Proposed: [0.08172258 0.83396588], UCB: 4.987215
  Best so far (Optuna): 6.112052e-01

Week 2:
  Actual observation: y = 5.728373e-01


[I 2026-04-23 19:22:48,201] Trial 34 finished with value: 4.275688403734796 and parameters: {'x0': 0.8493171329690629, 'x1': 0.08485852840470534}. Best is trial 23 with value: 4.6855388940514.
[I 2026-04-23 19:22:48,226] Trial 35 finished with value: 2.939418386722764 and parameters: {'x0': 0.7537008398476632, 'x1': 0.15549619614634286}. Best is trial 23 with value: 4.6855388940514.
[I 2026-04-23 19:22:48,234] Trial 36 finished with value: 2.8479840145870976 and parameters: {'x0': 0.9256772121614147, 'x1': 0.6423620561394953}. Best is trial 23 with value: 4.6855388940514.
[I 2026-04-23 19:22:48,240] Trial 37 finished with value: 1.995334239579973 and parameters: {'x0': 0.5228577143449558, 'x1': 0.24879358170004337}. Best is trial 23 with value: 4.6855388940514.
[I 2026-04-23 19:22:48,244] Trial 38 finished with value: 1.293991327058025 and parameters: {'x0': 0.662454575167001, 'x1': 0.1512975424122914}. Best is trial 23 with value: 4.6855388940514.
[I 2026-04-23 19:22:48,252] Trial 39 

[Iteration 0] Proposed: [0.9594353  0.27006861], UCB: 4.685714
  Best so far (Optuna): 6.112052e-01

Week 3:
  Actual observation: y = 4.997878e-01


...

[Iteration 0] Proposed: [0.9146073  0.78997937], UCB: 3.953112
  Best so far (Optuna): 6.112052e-01

Week 9:
  Actual observation: y = 5.670257e-02


[I 2026-04-23 19:22:53,362] Trial 22 finished with value: 3.834841178375836 and parameters: {'x0': 0.3961267582540022, 'x1': 0.642971971740546}. Best is trial 18 with value: 3.835710880580508.
[I 2026-04-23 19:22:53,369] Trial 23 finished with value: 3.834833817376812 and parameters: {'x0': 0.21690875688232988, 'x1': 0.4520608917725767}. Best is trial 18 with value: 3.835710880580508.
[I 2026-04-23 19:22:53,375] Trial 24 finished with value: 3.8348531324769772 and parameters: {'x0': 0.4704006374866203, 'x1': 0.7315845704025836}. Best is trial 18 with value: 3.835710880580508.
[I 2026-04-23 19:22:53,380] Trial 25 finished with value: 3.834833817376812 and parameters: {'x0': 0.36370035882090856, 'x1': 0.8096684738342856}. Best is trial 18 with value: 3.835710880580508.
[I 2026-04-23 19:22:53,385] Trial 26 finished with value: 3.8348423664119404 and parameters: {'x0': 0.5555587006111851, 'x1': 0.7239372357099974}. Best is trial 18 with value: 3.835710880580508.
[I 2026-04-23 19:22:53,391]

[Iteration 0] Proposed: [0.41610989 0.66699788], UCB: 3.841105
  Best so far (Optuna): 6.112052e-01

Week 10:
  Actual observation: y = 1.625184e-01


[I 2026-04-23 19:22:54,048] Trial 11 finished with value: 3.8363063916040403 and parameters: {'x0': 0.6473720400967767, 'x1': 0.5517130042066923}. Best is trial 0 with value: 3.8363063916040403.
[I 2026-04-23 19:22:54,054] Trial 12 finished with value: 3.8363063916040403 and parameters: {'x0': 0.715169612196795, 'x1': 0.3891958488372309}. Best is trial 0 with value: 3.8363063916040403.
[I 2026-04-23 19:22:54,058] Trial 13 finished with value: 3.8363165669295256 and parameters: {'x0': 0.47301636195212565, 'x1': 0.6379717552110609}. Best is trial 13 with value: 3.8363165669295256.
[I 2026-04-23 19:22:54,064] Trial 14 finished with value: 3.8363063916040434 and parameters: {'x0': 0.44168054887995184, 'x1': 0.01317673246334039}. Best is trial 13 with value: 3.8363165669295256.
[I 2026-04-23 19:22:54,070] Trial 15 finished with value: 3.8363063916040403 and parameters: {'x0': 0.5064602268978132, 'x1': 0.017193305737191855}. Best is trial 13 with value: 3.8363165669295256.
[I 2026-04-23 19:2

[Iteration 0] Proposed: [0.41610989 0.66699788], UCB: 3.842553
  Best so far (Optuna): 6.112052e-01

OPTUNA-BASED BO COMPLETED FOR FUNCTION 2
